# Libs

In [1]:
!apt-get update -y
!apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,721 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/u

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!pip install ollama pyngrok

In [4]:
import os
import ollama
from google.colab import userdata
from pyngrok import ngrok, conf
import requests

# Server

In [5]:
# !pkill -f ollama

In [6]:
!nohup bash -c "OLLAMA_HOST=0.0.0.0:8000 OLLAMA_KEEP_ALIVE=-1  OLLAMA_ORIGIN=* ollama serve" &
!sleep 5 && tail /content/nohup.out

nohup: appending output to 'nohup.out'
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIPuzMeHfjDrJNtrY99yDNGqvWC8SO+9qnt7MdnqYfAaO

time=2026-06-12T12:09:13.747Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:8000 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:2562047h47m16.854775807s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* http

# Models

In [11]:
os.environ["OLLAMA_HOST"] = "http://localhost:8000"

In [12]:
!ollama pull gemma4:e2b
!ollama pull qwen2.5:3b

# Models Calls

In [13]:
client = ollama.Client(host="http://localhost:8000")

In [14]:
def chat_with_llm(prompt: str):
    response = client.chat(
        model='qwen2.5:3b',
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]

In [15]:
print(chat_with_llm("What is Egypt? [only In 1 line]"))

Egypt is a country in the African continent, bordering Asia, located in the northeastern part of Africa.


# NGROK

In [16]:
NGROK_AUTH = userdata.get("colab-ngrok")
conf.get_default().auth_token = NGROK_AUTH

conf.get_default().log_level = "debug"
conf.get_default().monitor_thread = True

In [17]:
public_url = ngrok.connect(8000).public_url
print("Public Ollama URL:", public_url)

Public Ollama URL: https://68ea-35-233-154-11.ngrok-free.app


# Test

In [19]:
url = public_url + "/api/generate"

res = requests.post(url, json={
    "model": 'qwen2.5:3b',
    "prompt": "What is AI? [1 line answer]",
    "stream": False
})

print(res.json()["response"])

AI stands for Artificial Intelligence.


# Keep session busy

In [ ]:
while 1:
  print("\r.",end="")

.